In [2]:
import os
import glob
import random
import pandas as pd
from tqdm.auto import tqdm

tqdm.pandas()

In [7]:
# ============================================================
# Configuration
# ============================================================

# Folder containing all instruction CSV files
DATASET_FOLDER = "/kaggle/input/datasets/punitkashyap2007/virgo-instruction"

# Output folder
OUTPUT_FOLDER = "/kaggle/working/virgo_chat_dataset"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# Train/Validation split
TRAIN_RATIO = 0.98
RANDOM_SEED = 42

random.seed(RANDOM_SEED)

In [11]:
# ============================================================
# Load all CSV files (Robust Version)
# ============================================================

import csv

csv_files = sorted(glob.glob(os.path.join(DATASET_FOLDER, "*.csv")))

datasets = []

for file in csv_files:

    print(f"Loading {os.path.basename(file)}")

    try:
        df = pd.read_csv(
            file,
            engine="python",
            quoting=csv.QUOTE_MINIMAL,
            on_bad_lines="skip"
        )

        # lowercase columns
        df.columns = [c.strip().lower() for c in df.columns]

        # keep only first two columns
        df = df.iloc[:, :2]

        df.columns = ["prompt", "response"]

        df = df.dropna()

        df["prompt"] = df["prompt"].astype(str).str.strip()
        df["response"] = df["response"].astype(str).str.strip()

        df = df[
            (df["prompt"] != "") &
            (df["response"] != "")
        ]

        datasets.append(df)

        print(f"   ✓ {len(df):,} samples")

    except Exception as e:
        print(f"   ✗ {e}")

dataset = pd.concat(datasets, ignore_index=True)

print("\n==============================")
print("Total Samples :", len(dataset))
print("==============================")

dataset.head()

Loading exact_sentence_count.csv
   ✓ 10,025 samples
Loading exact_word_count.csv
   ✓ 10,161 samples
Loading markdown.csv
   ✓ 3,999 samples
Loading number_only.csv
   ✓ 8,043 samples
Loading one_sentence_ds.csv
   ✓ 9,996 samples
Loading one_word_ds.csv
   ✓ 10,134 samples
Loading virgo_identity.csv
   ✓ 5,520 samples
Loading yes_or_no.csv
   ✓ 8,116 samples

Total Samples : 65994


,prompt,response
0,Explain the solar system in exactly 2 sentences.,The solar system consists of the Sun and the o...
1,Describe photosynthesis in exactly 3 sentences.,Photosynthesis is the process by which green p...
2,What is gravity in exactly 1 sentence?,Gravity is the force that attracts objects wit...
3,Compare mammals and reptiles in exactly 4 sent...,Mammals are warm blooded animals while reptile...
4,Summarize the history of the printing press in...,The printing press transformed the spread of k...


In [75]:
import os

for file in csv_files:
    with open(file, "r", encoding="utf-8", errors="ignore") as f:
        total_lines = sum(1 for _ in f) - 1

    df = pd.read_csv(
        file,
        engine="python",
        quoting=csv.QUOTE_MINIMAL,
        on_bad_lines="skip"
    )

    print(
        f"{os.path.basename(file):30}"
        f" Expected: {total_lines:6}"
        f" Loaded: {len(df):6}"
        f" Lost: {total_lines-len(df):6}"
    )

exact_sentence_count.csv       Expected:  10095 Loaded:  10025 Lost:     70
exact_word_count.csv           Expected:  10215 Loaded:  10162 Lost:     53
markdown.csv                   Expected:   4026 Loaded:   3999 Lost:     27
number_only.csv                Expected:   8072 Loaded:   8044 Lost:     28
one_sentence_ds.csv            Expected:  10173 Loaded:   9996 Lost:    177
one_word_ds.csv                Expected:  10150 Loaded:  10134 Lost:     16
virgo_identity.csv             Expected:   5527 Loaded:   5524 Lost:      3
yes_or_no.csv                  Expected:   8117 Loaded:   8117 Lost:      0


In [12]:
# ============================================================
# Clean Dataset
# ============================================================

print("Before cleaning :", len(dataset))

# Remove exact duplicate prompt-response pairs
dataset = dataset.drop_duplicates(
    subset=["prompt", "response"]
)

# Remove duplicate prompts (keep first response)
dataset = dataset.drop_duplicates(
    subset=["prompt"],
    keep="first"
)

# Remove leading/trailing whitespace
dataset["prompt"] = dataset["prompt"].str.strip()
dataset["response"] = dataset["response"].str.strip()

# Remove empty rows
dataset = dataset[
    (dataset["prompt"].str.len() > 0) &
    (dataset["response"].str.len() > 0)
]

# Shuffle dataset
dataset = dataset.sample(
    frac=1,
    random_state=RANDOM_SEED
).reset_index(drop=True)

print("After cleaning :", len(dataset))

dataset.head()

Before cleaning : 65994
After cleaning : 61405


,prompt,response
0,Which database is developed by Oracle?,Oracle Database.
1,Describe briefly: What is the Southern Ocean?,The Southern Ocean surrounds Antarctica and co...
2,Outline the process of fossil fuel formation i...,Ancient plants and animals accumulated over mi...
3,Will your licensing always stay the same?,Not necessarily. Virgo is currently open sourc...
4,Choose Yes or No: Is JSON an operating system?,No


In [13]:
# ============================================================
# Convert Dataset to Virgo Chat Format
# ============================================================

BOS_TOKEN = "<bos>"
EOS_TOKEN = "<eos>"
NEWLINE_TOKEN = "<newline>"

def normalize(text):
    text = str(text).strip()

    # Normalize line endings
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Replace every newline with the special token
    text = text.replace("\n", NEWLINE_TOKEN)

    # Remove duplicate newline tokens
    while f"{NEWLINE_TOKEN}{NEWLINE_TOKEN}{NEWLINE_TOKEN}" in text:
        text = text.replace(
            f"{NEWLINE_TOKEN}{NEWLINE_TOKEN}{NEWLINE_TOKEN}",
            f"{NEWLINE_TOKEN}{NEWLINE_TOKEN}"
        )

    return text

def format_chat(prompt, response):
    prompt = normalize(prompt)
    response = normalize(response)

    return (
        f"{BOS_TOKEN}"
        f"User: {prompt}"
        f"{NEWLINE_TOKEN}{NEWLINE_TOKEN}"
        f"Assistant: {response}"
        f"{EOS_TOKEN}"
    )

dataset["text"] = dataset.apply(
    lambda row: format_chat(row["prompt"], row["response"]),
    axis=1
)

print(dataset["text"].iloc[0])

print(f"\nTotal Chat Samples: {len(dataset):,}")

<bos>User: Which database is developed by Oracle?<newline><newline>Assistant: Oracle Database.<eos>

Total Chat Samples: 61,405


In [14]:
# ============================================================
# Train / Validation Split
# ============================================================

from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    dataset[["text"]],
    test_size=1 - TRAIN_RATIO,
    random_state=RANDOM_SEED,
    shuffle=True
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"Training Samples   : {len(train_df):,}")
print(f"Validation Samples : {len(val_df):,}")

Training Samples   : 60,176
Validation Samples : 1,229


In [15]:
# ============================================================
# Load Virgo Tokenizer
# ============================================================

from tokenizers import Tokenizer

TOKENIZER_PATH = "/kaggle/input/datasets/punitkashyap2007/virgo-tokenizer/virgo_tokenizer.json"

tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

print("Tokenizer Loaded Successfully")
print("Vocabulary Size:", tokenizer.get_vocab_size())

Tokenizer Loaded Successfully
Vocabulary Size: 45000


In [16]:
# ============================================================
# Tokenize Dataset
# ============================================================

import numpy as np
from tqdm.auto import tqdm

OUTPUT_DIR = "/kaggle/working/virgo_chat_dataset"
os.makedirs(OUTPUT_DIR, exist_ok=True)


def encode_dataset(df, name):

    ids = []

    for text in tqdm(df["text"], desc=f"Encoding {name}"):

        encoded = tokenizer.encode(text)

        ids.extend(encoded.ids)

    ids = np.array(ids, dtype=np.uint16)

    output_path = os.path.join(OUTPUT_DIR, f"{name}.bin")
    ids.tofile(output_path)

    print(f"\n{name}.bin saved.")
    print(f"Tokens : {len(ids):,}")
    print(f"Size   : {ids.nbytes/1024/1024:.2f} MB")

    return ids


train_ids = encode_dataset(train_df, "train")
val_ids = encode_dataset(val_df, "val")

Encoding train:   0%|          | 0/60176 [00:00<?, ?it/s]


train.bin saved.
Tokens : 2,917,775
Size   : 5.57 MB


Encoding val:   0%|          | 0/1229 [00:00<?, ?it/s]


val.bin saved.
Tokens : 60,590
Size   : 0.12 MB


In [17]:
# ============================================================
# Verify Tokenized Dataset
# ============================================================

import numpy as np

train_ids = np.fromfile(
    os.path.join(OUTPUT_DIR, "train.bin"),
    dtype=np.uint16
)

val_ids = np.fromfile(
    os.path.join(OUTPUT_DIR, "val.bin"),
    dtype=np.uint16
)

print(f"Train Tokens : {len(train_ids):,}")
print(f"Validation Tokens : {len(val_ids):,}")

print("\nFirst 50 Train Tokens:")
print(train_ids[:50])

print("\nVocabulary Check")
print("Minimum Token ID :", train_ids.min())
print("Maximum Token ID :", train_ids.max())

assert train_ids.max() < tokenizer.get_vocab_size(), "Found invalid token!"
assert val_ids.max() < tokenizer.get_vocab_size(), "Found invalid token!"

print("\n✅ Dataset verification passed.")

Train Tokens : 2,917,775
Validation Tokens : 60,590

First 50 Train Tokens:
[    2 12187   217    31 21646 17603 11862   289  3411   212  1125 18655
    19     4     4 18772  8225    31  3942 10230  8864 29768   288 17603
    65    83    65    83     8     8  6069  6432    65    83    65    83
    69    53 21436  2850   475    78    20    89   807  1521    69    65
    83    65]

Vocabulary Check
Minimum Token ID : 0
Maximum Token ID : 44997

✅ Dataset verification passed.


In [18]:
# ============================================================
# Save Dataset Information
# ============================================================

import json

dataset_info = {
    "train_samples": len(train_df),
    "validation_samples": len(val_df),
    "train_tokens": int(len(train_ids)),
    "validation_tokens": int(len(val_ids)),
    "vocab_size": tokenizer.get_vocab_size(),
    "train_file": "train.bin",
    "validation_file": "val.bin"
}

with open(os.path.join(OUTPUT_DIR, "dataset_info.json"), "w") as f:
    json.dump(dataset_info, f, indent=4)

print(json.dumps(dataset_info, indent=4))

{
    "train_samples": 60176,
    "validation_samples": 1229,
    "train_tokens": 2917775,
    "validation_tokens": 60590,
    "vocab_size": 45000,
    "train_file": "train.bin",
    "validation_file": "val.bin"
}


In [19]:
# ============================================================
# Decode a Sample
# ============================================================

sample_ids = train_ids[:200]

decoded = tokenizer.decode(sample_ids.tolist())

print(decoded)

User: Prepare API documentation for creating a support ticket.Assistant: # Create Support Ticket API\n\n## Endpoint\n\n`POST /api/tickets`\n\n## Request Body\n\n| Field | Type | Required |\n|---|---|:---:|\n| subject | string | ✓ |\n| description | string | ✓ |\n| priority | string | ✗ |\n\n## Response\n\nReturns the created support ticket.User: Could you tell me how genetics influence behavior? Explain in exactly 100 words.Assistant: Genetics influence behavior by contributing biological factors that affect personality abilities emotions and psychological characteristics. Researchers study how inherited traits interact with environmental experiences to shape individual differences. Genetics alone does not determine behavior because learning culture and relationships also play important roles. Psychology examines the interaction


In [20]:
# ============================================================
# GPT Dataset
# ============================================================

import torch
from torch.utils.data import Dataset

BLOCK_SIZE = 1024

class VirgoDataset(Dataset):

    def __init__(self, tokens, block_size):
        self.tokens = torch.tensor(tokens, dtype=torch.long)
        self.block_size = block_size

    def __len__(self):
        return len(self.tokens) - self.block_size

    def __getitem__(self, idx):

        x = self.tokens[idx:idx + self.block_size]

        y = self.tokens[idx + 1:idx + self.block_size + 1]

        return x, y


train_dataset = VirgoDataset(train_ids, BLOCK_SIZE)
val_dataset = VirgoDataset(val_ids, BLOCK_SIZE)

print("Train Sequences :", len(train_dataset))
print("Validation Sequences :", len(val_dataset))

Train Sequences : 2916751
Validation Sequences : 59566


In [51]:
class CFG:

    # ==========================================================
    # Dataset
    # ==========================================================

    train_bin = "/kaggle/working/virgo_chat_dataset/train.bin"
    val_bin   = "/kaggle/working/virgo_chat_dataset/val.bin"

    tokenizer_path = "/kaggle/input/datasets/punitkashyap2007/virgo-tokenizer/virgo_tokenizer.json"

    # ==========================================================
    # Checkpoint
    # ==========================================================

    checkpoint = "/kaggle/input/datasets/punitkashyap2007/virgo-chat-model/virgo_chat_best_h100_ep2.pt"

    output_dir = "/kaggle/working/virgo_chat_v2"

    # ==========================================================
    # Model (UNCHANGED)
    # ==========================================================

    vocab_size = 45000

    d_model = 768
    num_heads = 12
    num_layers = 12
    d_ff = 3072

    max_seq_length = 1024

    dropout = 0.1

    # ==========================================================
    # Fine-tuning
    # ==========================================================

    epochs = 3

    batch_size = 4

    gradient_accumulation = 32

    learning_rate = 1e-5

    min_lr = 1e-6

    weight_decay = 0.01

    warmup_ratio = 0.03

    grad_clip = 1.0

    num_workers = 2

    seed = 42

In [52]:
# ==========================================================
# Load Binary Token Files
# ==========================================================

import numpy as np

train_tokens = np.memmap(
    CFG.train_bin,
    dtype=np.uint16,
    mode="r"
)

val_tokens = np.memmap(
    CFG.val_bin,
    dtype=np.uint16,
    mode="r"
)

print(f"Train Tokens      : {len(train_tokens):,}")
print(f"Validation Tokens : {len(val_tokens):,}")

Train Tokens      : 2,917,775
Validation Tokens : 60,590


In [53]:
from torch.utils.data import Dataset
import numpy as np
import torch


class BinaryDataset(Dataset):

    def __init__(self, bin_file, seq_len):

        self.seq_len = seq_len

        self.data = np.memmap(
            bin_file,
            dtype=np.uint16,
            mode="r"
        )

        # Number of COMPLETE sequences only
        self.num_sequences = (len(self.data) - seq_len - 1) // seq_len

    def __len__(self):
        return self.num_sequences

    def __getitem__(self, idx):

        start = idx * self.seq_len
        end = start + self.seq_len

        x = torch.tensor(
            self.data[start:end],
            dtype=torch.long
        )

        y = torch.tensor(
            self.data[start + 1:end + 1],
            dtype=torch.long
        )

        return x, y

In [54]:
train_dataset = BinaryDataset(
    bin_file=CFG.train_bin,
    seq_len=CFG.max_seq_length
)

val_dataset = BinaryDataset(
    bin_file=CFG.val_bin,
    seq_len=CFG.max_seq_length
)

In [55]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=CFG.batch_size,
    shuffle=True,
    num_workers=CFG.num_workers,
    pin_memory=True,
    drop_last=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CFG.batch_size,
    shuffle=False,
    num_workers=CFG.num_workers,
    pin_memory=True,
    drop_last=False,
)

print(f"Train sequences : {len(train_dataset):,}")
print(f"Validation sequences : {len(val_dataset):,}")

print(f"Train batches : {len(train_loader):,}")
print(f"Validation batches : {len(val_loader):,}")

Train sequences : 2,848
Validation sequences : 58
Train batches : 712
Validation batches : 15


In [56]:
import torch
from tokenizers import Tokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

tokenizer = Tokenizer.from_file(CFG.tokenizer_path)

print("Vocabulary Size :", tokenizer.get_vocab_size())
print("Checkpoint Path :", CFG.checkpoint)

Device: cuda
Vocabulary Size : 45000
Checkpoint Path : /kaggle/input/datasets/punitkashyap2007/virgo-chat-model/virgo_chat_best_h100_ep2.pt


In [57]:
class RotaryEmbedding(nn.Module):
    def __init__(self, d_k, max_seq_length=2048, base=10000.0):
        super(RotaryEmbedding, self).__init__()

        assert d_k % 2 == 0, "Head dimension must be even for RoPE"

        inv_freq = 1.0 / (base ** (torch.arange(0, d_k, 2).float() / d_k))
        positions = torch.arange(max_seq_length).float()

        freqs = torch.outer(positions, inv_freq)

        self.register_buffer("cos", freqs.cos()[None, None, :, :], persistent=False)
        self.register_buffer("sin", freqs.sin()[None, None, :, :], persistent=False)

    def forward(self, Q, K):
        seq_length = Q.size(-2)

        cos = self.cos[:, :, :seq_length, :].to(dtype=Q.dtype)
        sin = self.sin[:, :, :seq_length, :].to(dtype=Q.dtype)

        Q_even = Q[..., 0::2]
        Q_odd = Q[..., 1::2]

        K_even = K[..., 0::2]
        K_odd = K[..., 1::2]

        Q = torch.stack((Q_even * cos - Q_odd * sin, Q_even * sin + Q_odd * cos), dim=-1).flatten(-2)
        K = torch.stack((K_even * cos - K_odd * sin, K_even * sin + K_odd * cos), dim=-1).flatten(-2)

        return Q, K

In [58]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, max_seq_length):
        super(MultiHeadAttention, self).__init__()

        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

        self.rope = RotaryEmbedding(self.d_k, max_seq_length)

    def split_heads(self, x):
        batch_size, seq_length, _ = x.size()
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)

    def combine_heads(self, x):
        batch_size, _, seq_length, _ = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_model)

    def forward(self, x):
        Q = self.split_heads(self.W_q(x))
        K = self.split_heads(self.W_k(x))
        V = self.split_heads(self.W_v(x))

        Q, K = self.rope(Q, K)

        attn_output = F.scaled_dot_product_attention(Q, K, V, dropout_p=0.0, is_causal=True)

        return self.W_o(self.combine_heads(attn_output))

In [59]:
class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PositionWiseFeedForward, self).__init__()

        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.gelu = nn.GELU()

    def forward(self, x):
        return self.fc2(self.gelu(self.fc1(x)))

In [60]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, max_seq_length, dropout):
        super(TransformerBlock, self).__init__()

        self.self_attn = MultiHeadAttention(d_model, num_heads, max_seq_length)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        norm_x = self.norm1(x)
        x = x + self.dropout(self.self_attn(norm_x))

        norm_x = self.norm2(x)
        x = x + self.dropout(self.feed_forward(norm_x))

        return x

In [61]:
class VirgoModel(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_length, dropout):
        super(VirgoModel, self).__init__()

        self.vocab_size = vocab_size
        self.d_model = d_model
        self.max_seq_length = max_seq_length

        self.token_embedding = nn.Embedding(vocab_size, d_model)

        self.layers = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff, max_seq_length, dropout)
            for _ in range(num_layers)
        ])

        self.norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        self.dropout = nn.Dropout(dropout)

        self.lm_head.weight = self.token_embedding.weight

    def forward(self, input_ids):
        seq_length = input_ids.size(1)

        if seq_length > self.max_seq_length:
            raise ValueError(f"Sequence length {seq_length} exceeds maximum {self.max_seq_length}")

        x = self.token_embedding(input_ids)
        x = self.dropout(x)

        for layer in self.layers:
            x = layer(x)

        x = self.norm(x)

        return self.lm_head(x)

In [62]:
# ==========================================================
# Build Model
# ==========================================================

model = VirgoModel(
    vocab_size=CFG.vocab_size,
    d_model=CFG.d_model,
    num_heads=CFG.num_heads,
    num_layers=CFG.num_layers,
    d_ff=CFG.d_ff,
    max_seq_length=CFG.max_seq_length,
    dropout=CFG.dropout,
)

model = model.to(device)

print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

Parameters: 119,616,000


In [63]:
# ==========================================================
# Load Pretrained Checkpoint
# ==========================================================

checkpoint = torch.load(CFG.checkpoint, map_location=device)

# If the checkpoint contains extra information
if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    state_dict = checkpoint["model_state_dict"]
else:
    state_dict = checkpoint

missing_keys, unexpected_keys = model.load_state_dict(
    state_dict,
    strict=False
)

print("✅ Checkpoint Loaded")

print(f"Missing Keys    : {len(missing_keys)}")
print(f"Unexpected Keys : {len(unexpected_keys)}")

if len(missing_keys):
    print("\nMissing Keys:")
    for k in missing_keys:
        print(k)

if len(unexpected_keys):
    print("\nUnexpected Keys:")
    for k in unexpected_keys:
        print(k)

✅ Checkpoint Loaded
Missing Keys    : 0
Unexpected Keys : 0


In [64]:
# ==========================================================
# Optimizer & Scheduler
# ==========================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG.learning_rate,
    weight_decay=CFG.weight_decay
)

total_steps = len(train_loader) * CFG.epochs
warmup_steps = int(total_steps * CFG.warmup_ratio)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=total_steps,
    eta_min=CFG.min_lr
)

criterion = torch.nn.CrossEntropyLoss()

print(f"Total Steps  : {total_steps}")
print(f"Warmup Steps : {warmup_steps}")
print(f"Learning Rate: {CFG.learning_rate}")

Total Steps  : 2136
Warmup Steps : 64
Learning Rate: 1e-05


In [65]:
print("Model device:", next(model.parameters()).device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total Parameters     : {total_params:,}")
print(f"Trainable Parameters : {trainable_params:,}")

Model device: cuda:0
Total Parameters     : 119,616,000
Trainable Parameters : 119,616,000


In [66]:
# ==========================================================
# Sanity Check
# ==========================================================

model.train()

x, y = next(iter(train_loader))

x = x.to(device)
y = y.to(device)

with torch.cuda.amp.autocast():
    logits = model(x)

    loss = criterion(
        logits.view(-1, logits.size(-1)),
        y.view(-1)
    )

print("Input Shape :", x.shape)
print("Target Shape:", y.shape)
print("Logits Shape:", logits.shape)
print("Loss:", loss.item())

Input Shape : torch.Size([4, 1024])
Target Shape: torch.Size([4, 1024])
Logits Shape: torch.Size([4, 1024, 45000])
Loss: 5.09135627746582


In [67]:
# ==========================================================
# Fine-tuning Loop
# ==========================================================

from tqdm.auto import tqdm

scaler = torch.cuda.amp.GradScaler()

best_loss = float("inf")

for epoch in range(CFG.epochs):

    model.train()

    running_loss = 0.0

    progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CFG.epochs}")

    for step, (x, y) in enumerate(progress):

        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast():

            logits = model(x)

            loss = criterion(
                logits.view(-1, logits.size(-1)),
                y.view(-1)
            )

        scaler.scale(loss).backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        scaler.step(optimizer)
        scaler.update()

        scheduler.step()

        running_loss += loss.item()

        progress.set_postfix(
            loss=f"{running_loss/(step+1):.4f}",
            lr=f"{scheduler.get_last_lr()[0]:.2e}"
        )

    epoch_loss = running_loss / len(train_loader)

    print(f"\nEpoch {epoch+1}")
    print(f"Training Loss: {epoch_loss:.4f}")

    if epoch_loss < best_loss:

        best_loss = epoch_loss

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "epoch": epoch,
                "loss": best_loss,
            },
            "virgo_instruction_best.pt",
        )

        print("✅ Best model saved.")

Epoch 1/3:   0%|          | 0/712 [00:00<?, ?it/s]


Epoch 1
Training Loss: 2.7224
✅ Best model saved.


Epoch 2/3:   0%|          | 0/712 [00:00<?, ?it/s]


Epoch 2
Training Loss: 2.3964
✅ Best model saved.


Epoch 3/3:   0%|          | 0/712 [00:00<?, ?it/s]

Exception ignored in: Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7a9a44181080><function _MultiProcessingDataLoaderIter.__del__ at 0x7a9a44181080>

Traceback (most recent call last):
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
        self._shutdown_workers()
self._shutdown_workers()  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers

      File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
if w.is_alive():    
if w.is_alive(): 
           ^^ ^^ ^^^^^^^^^^^^^^^^^^
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    ^assert self._parent_pid == os.getpid(), 'can only test a child process'

  File "/usr/lib/python


Epoch 3
Training Loss: 2.3423
✅ Best model saved.


In [73]:
import torch

# ==========================================================
# Load Fine-tuned Model
# ==========================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = VirgoModel(
    vocab_size=CFG.vocab_size,
    d_model=CFG.d_model,
    num_heads=CFG.num_heads,
    num_layers=CFG.num_layers,
    d_ff=CFG.d_ff,
    max_seq_length=CFG.max_seq_length,
    dropout=0.0,
).to(device)

checkpoint = torch.load(
    "virgo_instruction_best.pt",
    map_location=device
)

if "model_state_dict" in checkpoint:
    model.load_state_dict(checkpoint["model_state_dict"])
else:
    model.load_state_dict(checkpoint)

model.eval()

print("✅ Model Loaded")


# ==========================================================
# Inference
# ==========================================================

def generate(
    user_prompt,
    max_new_tokens=256,
    temperature=0.8,
    top_k=40,
    top_p=0.95,
    repetition_penalty=1.1,
):

    # Same format used during training
    prompt = (
        "<bos>"
        f"User: {user_prompt}"
        "<newline><newline>"
        "Assistant:"
    )

    encoding = tokenizer.encode(prompt)
    input_ids = torch.tensor(
        [encoding.ids],
        dtype=torch.long,
        device=device,
    )

    prompt_len = input_ids.size(1)

    eos_id = tokenizer.token_to_id("<eos>")

    with torch.no_grad():

        for _ in range(max_new_tokens):

            x = input_ids[:, -CFG.max_seq_length:]

            logits = model(x)
            logits = logits[:, -1, :]

            # repetition penalty
            for token in set(input_ids[0].tolist()):
                logits[0, token] /= repetition_penalty

            logits /= temperature

            # Top-k
            if top_k > 0:
                values, indices = torch.topk(logits, top_k)
                filtered = torch.full_like(logits, float("-inf"))
                filtered.scatter_(1, indices, values)
                logits = filtered

            probs = torch.softmax(logits, dim=-1)

            # Top-p
            sorted_probs, sorted_indices = torch.sort(
                probs,
                descending=True
            )

            cumulative = torch.cumsum(sorted_probs, dim=-1)

            remove = cumulative > top_p
            remove[..., 1:] = remove[..., :-1].clone()
            remove[..., 0] = False

            sorted_probs[remove] = 0
            sorted_probs /= sorted_probs.sum()

            next_token = sorted_indices.gather(
                1,
                torch.multinomial(sorted_probs, 1)
            )

            input_ids = torch.cat(
                [input_ids, next_token],
                dim=1
            )

            if eos_id is not None and next_token.item() == eos_id:
                break

    # Decode ONLY generated tokens
    generated_ids = input_ids[0].tolist()[prompt_len:]

    response = tokenizer.decode(generated_ids)

    response = response.replace("<newline>", "\n")
    response = response.replace("<eos>", "")
    response = response.replace("<bos>", "")
    response = response.strip()

    return response

✅ Model Loaded


In [74]:
while True:

    user = input("\nYou: ")

    if user.lower() in ["exit", "quit"]:
        break

    print("\nVirgo:", generate(user))


You:  hi



Virgo: Can you explain how technology works?



You:  introduce yourself



Virgo: Welcome to my portfolio of engineering and digital tools.



You:  who is your creator


...


You:  what is gravity in 1 sentence ?



Virgo: Gravity has the potential to create both gravitational forces and gravity.



You:  what is gravity explain in 100 words.



Virgo: Gravity explains the motion of planets with gravity by analyzing how energy changes on surfaces and in gravitational interactions between planets.



You:  exit


In [76]:
print("bos:", tokenizer.token_to_id("<bos>"))
print("eos:", tokenizer.token_to_id("<eos>"))
print("newline:", tokenizer.token_to_id("<newline>"))

bos: 2
eos: 3
newline: 4


In [77]:
queries = [
    "Who created Virgo?",
    "Who is your creator?",
    "Who developed Virgo?",
    "Who is Punit Kumar Kashyap?",
    "Introduce yourself"
]

for q in queries:
    rows = dataset[
        dataset["prompt"].str.lower() == q.lower()
    ]

    print("=" * 80)
    print(q)
    print(rows.head(5))

Who created Virgo?
Empty DataFrame
Columns: [prompt, response, text]
Index: []
Who is your creator?
                     prompt  \
46337  Who is your creator?   

                                                response  \
46337  My creator is Punit Kumar Kashyap, a student a...   

                                                    text  
46337  <bos>User: Who is your creator?<newline><newli...  
Who developed Virgo?
                     prompt  \
29556  Who developed Virgo?   

                                                response  \
29556  Virgo was developed by Punit Kumar Kashyap, a ...   

                                                    text  
29556  <bos>User: Who developed Virgo?<newline><newli...  
Who is Punit Kumar Kashyap?
                            prompt  \
23226  Who is Punit Kumar Kashyap?   

                                                response  \
23226  Punit Kumar Kashyap is an Engineering Physics ...   

                                                 

In [78]:
print("Total samples:", len(dataset))
print("Unique prompts:", dataset["prompt"].nunique())
print("Duplicate prompts:", len(dataset) - dataset["prompt"].nunique())

Total samples: 61405
Unique prompts: 61405
Duplicate prompts: 0


In [79]:
print(checkpoint.keys())

dict_keys(['model_state_dict', 'optimizer_state_dict', 'epoch', 'loss'])
